Imports:

In [1]:
#!/usr/bin/env python3
import numpy as np
from math import sqrt, pi
from collections import defaultdict
from dataclasses import dataclass
from scipy.sparse import dok_matrix, csr_matrix
from scipy.sparse.linalg import eigsh

##### Finite Volume and momentum modes
Quantise omega on a circle with radius R:
$$\omega = \sqrt{(\frac{k}{R})^2 + m_q^2}$$
Free Hamiltonian:
$$H_0 = \sum_k \omega_k a^{\dagger}_k a_k$$

build_modes is a practical momentum cutoff, essentially implements a finite k max

In [2]:
def omega(k: int, R: float, mQ: float) -> float:
    return sqrt((k / R) ** 2 + mQ ** 2)

def build_modes(kmax: int):
    return list(range(-kmax, kmax + 1))

##### Making the basis under $H_0$ and $E_{max}$

Define a function that builds the truncated basis

inputs
```modes``` is a list of allowed momentums, ```omegas``` is a dictionary mapping k to omega (mode frequencies), ```Emax``` is the truncation energy

```p_total``` is the desired total momentum sector
- our field is defined on circle of radius R
- total momentum must be conserved

In momentum space:
$$P = \sum_k k n_k$$

where k is discrete momemntum modes and $n_k$ is occupation number of mode k
Therefore P here is the eigenvalue of the total momentum operator $\hat{P}$ acting on a Fock state (our energy states). therefore by defining p_total = 0 we keep only states with zero net momentum, in the future we could change it to something like 1 or -1 and that would keep only states with that total momentum but because our lowest energy states have zero total momentum we use 0 as P=0 contains all relevant states. Looking at this another way in an infinite volume the vacuum is translation invariant, the analogue of translation invariant  in a finite volume is zero total momentum so by choosing p_total = 0 it corresponds to a physical vacuum sector.

THen enforce_p0_sector = true is a switch that decides whether momentum conservation is enforced at the level of the basis, ```True``` gives 
$$\sum_k k n_k = p_{total}$$

and ```False``` gives all momentum sectors included i.e the basis contains states with many different total momenta

we can justify this by looking at $H = H_0 + V$ as $[H,\hat{P}]=0$ so we know states with different total momentum never mix and the H matrix is block diagonal with one block per momentum sector, i.e working in a fixed P sector is exact

by also doing this we reduce the number of states, no momentum restriction leads to thousands of states whereas P = 0 only reduces it by a few orders of magnitude

if we choose not to enforce the limit the hamiltonian is still block diagonal however using eigsh will waste effort diagonalising a much larger matrix, the only time youll want to switch this off is:
- studying dispersion relations E(P)
- Testing momemntum conservation numerically


In [3]:
def generate_basis(modes, omegas, Emax, p_total=0, enforce_p0_sector=True):

    #sort modes by increasing omega
    #basically just for efficiency as low omega modes allow more occupancy
    mode_order = sorted(modes, key=lambda k: omegas[k]) 
    basis = [] # for valid basis states (tuple of occupations in order of modes)
    n_dict = {k: 0 for k in modes} #occupation number that are filled out during recursion

    #recursive function that decides occupation mode by mode
    #i - which mode in mode order we're gonna use
    #E-used current accumulated free energy
    #P_used current accumulated momentum
    def rec(i, E_used, P_used):
        if E_used > Emax + 1e-12: # checks if we already exceed the energy budget
            return
        if i == len(mode_order): # just checks theres an occupattion number for every mode
            #if we're not enforcing momenutm, we accept the state
            #if we are enforcing it, we check if the total momentum equal p_total
            if (not enforce_p0_sector) or (P_used == p_total): 
                basis.append(tuple(n_dict[k] for k in modes)) #stores it in the basis 
            return #returns after processing the fully assigned state

        #chooses next modes occupation:
        #selects next mode k to assign and its frequency
        k = mode_order[i] 
        w = omegas[k] 

        #computes maximum occupation number can place in next mode without exceeding the cutoff
        nmax = int((Emax - E_used) // w) if w > 0 else 0 #note use of floor division here //
        
        
        for n in range(nmax + 1): #loopo over allowed occupations for mode
            n_dict[k] = n #sets current modes occupation to n in working
            rec(i + 1, E_used + n * w, P_used + k * n) #recursive to next mode, updates energy and momentum by n * omega and k * n
        n_dict[k] = 0 #cleans up dict after loop so we dont mess up recursion next time

    rec(0, 0.0, 0) #starts recursion at 0, 0, 0
    return basis # returns all basis states meeting the truncation constraints we set out

##### Creation and Annhiliation Operators

standard fock space algebra:

$$ a_{k}\ket{...,n_k,...}=\sqrt{n_k}\ket{...,n_k - 1,...} $$


$$ a_{k}^\dagger \ket{...,n_k,...}=\sqrt{n_k +1}\ket{...,n_k + 1,...} $$



In [4]:
def apply_annihilate(state, modes, mode_index, k): #applies operator to basis vecotr
    idx = mode_index[k] #looks up which entry of a tupple corresponds to mode
    n = state[idx] # reads occupation number for mode
    if n == 0:
        return 0.0, None #no resulting state
    new = list(state)
    new[idx] -= 1 # reduces occupation by 1
    return sqrt(n), tuple(new) #returns prefactor and new state

#same as above but for creation so +1 instead of -1
def apply_create(state, modes, mode_index, k):
    idx = mode_index[k]
    n = state[idx]
    new = list(state)
    new[idx] += 1
    return sqrt(n + 1), tuple(new)

##### $\phi_k$ in terms of ladder operators

input:
```state``` is the occupation tuple, ```modes``` is a list of mode labels in  order, ```mode_index``` is a dict which maps k to an index in the tuple, ```omegas``` is a dict which maps k to omega, ```L``` is spatial length, ```k``` is the omemntum mode operator which we're applying phi to

we're applying 

$$\phi_k \approx \frac{1}{\sqrt{2\omega_{k}L}} (a_k + a^{\dagger}_{k})$$

in the interaction builder we end up with four $ \phi_{k_i} $ in sequence and each splits into 2 branches which we accumulate into the matrix element, so the function we define returns a list as its designed to be used repeatedly 


In [5]:
def apply_phi_k(state, modes, mode_index, omegas, L, k):
    wk = omegas[k] #look up mode frequency for k
    norm = 1.0 / sqrt(2.0 * wk * L) #computes prefactor

    out = [] #output list

    # a_k term
    #c1 is ladder coffecient sqrt(n_k)
    #s1 is the new state n_k -> n_k-1
    c1, s1 = apply_annihilate(state, modes, mode_index, k)
    if s1 is not None: #checks if annhiliation was possible
        out.append((norm * c1, s1))

    # a_{-k}^\dagger term
    #similar as above for c2 and s2
    c2, s2 = apply_create(state, modes, mode_index, -k)
    out.append((norm * c2, s2))

    return out #get out a superposition


##### Construction of interaction matrix V

We want to fill out:

$$V_{ji} = \bra{j}V\ket{i}$$

with

$$V = \frac{\lambda}{4!} \int dx :\phi^4:$$

by acting with the momentum space operator product $\phi_{k_1}\phi_{k_2}\phi_{k_3}\phi_{k_4}$ on each $\ket{i}$ and enforcing $k_1 + k_2 + k_3 + k_4 = 0$ then project the result into truncated basis

inputs:
```basis``` is list of occupation tuples $\ket = (n_{k_1},...,n_{k_m})$, ```basis_index``` dict mapping the state tuple to an index j, ```modes``` is a list of allowed momenta k, ```omegas``` dict that maps k to omega, ```L``` is a spatial length, ```lam``` coupling lamda 

In [6]:
def build_V_phi4(basis, basis_index, modes, omegas, L, lam):
    dim = len(basis) #truncated basis dimension
    mode_index = {k: i for i, k in enumerate(modes)} #maps each k to position in tuple
    V = dok_matrix((dim, dim), dtype=np.float64) #basically just an efficient matrix we use for V

    # Precompute all quartets
    # satisfies k1+k2+k3+k4=0
    # probably not the most efficient way of doing this
    # NOTE this means that permutations appear multiple times:
    # this isnt inherently wrong but needs to be accounted for in normalisation
    quartets = []
    for k1 in modes:
        for k2 in modes:
            for k3 in modes:
                k4 = -(k1 + k2 + k3)
                if k4 in mode_index:
                    quartets.append((k1, k2, k3, k4))

    pref = lam / 24.0  #coupling factor (nice and easy)

    for i, ket in enumerate(basis): #loop over each basis state and apply V
        accum = defaultdict(float) #collects contributions to different basis states before applying to V
        #^important because quartets and operator branchings can land on same final state so we sum them

        #for each quartet we apply our phik1phik2phik3phik4
        #again this definitely is efficient at all
        for (k1, k2, k3, k4) in quartets:
            terms1 = apply_phi_k(ket, modes, mode_index, omegas, L, k1) #list of coefficient and state pairs for phik1
            for c1, s1 in terms1: #loops over each branch
                terms2 = []
                for c, s in apply_phi_k(s1, modes, mode_index, omegas, L, k2): #apply phik2 to each intermediate state s1
                    terms2.append((c1 * c, s)) #coefficient after two operators
                    #we store in terms2 as its convenient to carry forward combined coefficient and states
                for c12, s2 in terms2: #same logic again, apply, accumulate combined coefficinet
                    terms3 = []
                    for c, s in apply_phi_k(s2, modes, mode_index, omegas, L, k3):
                        terms3.append((c12 * c, s))
                    for c123, s3 in terms3: # finally apply 4th phi giving final states (bra_state with coefficient c)
                        for c, bra_state in apply_phi_k(s3, modes, mode_index, omegas, L, k4):
                            ctot = c123 * c #total amplitude
                            # Project back into truncated basis if present
                            j = basis_index.get(bra_state, None)
                            if j is not None: #check if bra_state is in truncated basis, if not discard (truncation), if yes mapped to index j and added to accum
                                accum[j] += ctot

        for j, val in accum.items():
            V[j, i] += pref * val #accumulated contributions written into matrix

    return V.tocsr() #converted to csr (as its faster for matrix addition and matrix vector products inside eigsh)

#outputted a sparese matrix V representing interacting part of the Hamiltonian in the truncated basis 
# this is really expensive (O(N^3))
# definitely needs relooking at where we dont overcount

##### Single number HTET correction (a "matching" correction)

Use this to shift the quartic coupling $\lambda \to \lambda_{eff} = \lambda + \lambda_2$. States above truncation cutoff $E_{max}$ are not included explicitely so approximate effect by adding correction to couplings in the effective hamiltonian

inputs:
```lam``` bare coupling lambda, ```R``` circle radius, ```omegas``` dict mapping momemntum k to omega, ```modes``` list of momentum labels k, ```Emax``` truncation energy cutoff

final output we want is

$$\lambda_2 = -\frac{3\lambda^2}{16\pi R}\sum_{k~within~modes}\Theta(2\omega_k-E_{max})\frac{1}{\omega_k^3}$$

This is the big part of HTET, when we truncate at E_max we remove high energy states but those states still affect low energy observables through virtual processes, HTET says instead of including those states we approximate their effects by adding local terms to an effective H acting in low energy subspace. One consequence is that the quartic interaction strength gets renormalised as above

we compute $\lambda_2$ from cutoff and free spectrum, its absorbed into an improved effective coupling, and then we rebuild the interaction with $\lambda_{eff}$

Caveat for interpretation:
function is a model of the leading correction, the exact numerical prefactor and whether it should be treated purely as a shift of $\lambda$ is dependent on the normalisation conventions of $\phi_{k}$ and integrals, which operators we keep (HTET generally induces more than just a phi^4 term) and then the dimensionality used in the derivation we match to

In [7]:
def lambda2_htet(lam, R, omegas, modes, Emax):
    s = 0.0 #initialises sum that'll be on the RHS
    for k in modes: #loops over all momentum modes
        if 2.0 * omegas[k] > Emax: #essentially a step function
            #mode k only contributes if 2 particle enrgy 2omega lies above truncation
            #when getting second order corrections, sum over intermediate states that are outside truncated space
            #for two particle intermediate state with k, free energy is 2omega
            #this condition selects the ones that the cutoff excludes
            s += 1.0 / (omegas[k] ** 3) # adds contribution of a given mode
            #omega^-3 dependence comes from sum that appears in leading O(lambda^2) correction for phi^4
    return -(3.0 * lam * lam) / (16.0 * pi * R) * s

##### Solver Class 

Fixes parameters
Choose a truncation
Generates Basis
Build H0
Diagonalise H to get low energy levels

Can then repeat for different E_max to see convergence and compare no improved coupling vs improved


Params:
```R``` circle radius
```mQ``` quantisation masss (used in the free disperion of omega)
```lam``` the coupling coefficient we use
```kmax``` momentum mode cut off


In [8]:
@dataclass
class Params:
    R: float
    mQ: float
    lam: float
    kmax: int

class HTETSolver:
    #sets up every defining the truncated problem given Emax, geometry (L), allowed momenta, free frequencies, truncated basis and indexing
    def __init__(self, params: Params, Emax: float, enforce_p0_sector=True):  
        self.p = params #stores R, mQ, lambda and kmax
        self.Emax = Emax #sotres truncation energy cutoff

        self.L = 2.0 * pi * self.p.R #circumfrence L
        self.modes = build_modes(self.p.kmax) #gets our build_modes function which returns -kmax to kmax
        self.omegas = {k: omega(k, self.p.R, self.p.mQ) for k in self.modes} #builds dictionary mapping each k to free frequency

        #gets the basis (a list of tuples of n from -kmax to kmax) such that energy cutoff and momentum sector is enforced (if true)
        self.basis = generate_basis(self.modes, self.omegas, Emax, p_total=0, enforce_p0_sector=enforce_p0_sector)
        self.basis_index = {st: i for i, st in enumerate(self.basis)} #builds a reverse look up dictionary

    #build H0 in truncated basis
    def build_H0(self):
        dim = len(self.basis) #dimension of basis
        H0 = dok_matrix((dim, dim), dtype=np.float64) #same as our V previously just a matrix we can use in DOK
        for i, st in enumerate(self.basis): #loops over basis states with index i, each st is a tuple of occupations
            E = 0.0 #energy accumulator for this basis state
            for idx, k in enumerate(self.modes): #loops over all k 
                E += st[idx] * self.omegas[k] #adds to the energy
            H0[i, i] = E #diagonal element of H0 for basis state i to free energy
        return H0.tocsr() #again convert back to CSR as faster for matrix operations + eigensolving

    def build_H(self, improved=False): #construct full H matrix, imrpoved here toggles whether we replace lambda by an effective coupling
        lam_eff = self.p.lam #effective coupling equals input coupling by default
        
        if improved: #if we do want the improved we add the lambda correction
            lam_eff = self.p.lam + lambda2_htet(self.p.lam, self.p.R, self.omegas, self.modes, self.Emax) 

        H0 = self.build_H0() #builds h0
        #gets interaction matrix
        V = build_V_phi4(self.basis, self.basis_index, self.modes, self.omegas, self.L, lam_eff) 

        return (H0 + V).tocsr(), lam_eff #adds together the matrices  and reforces CSR

    def solve_lowest(self, n_eigs=5, improved=False): # compute our lowest eigenvalues of H
        H, lam_eff = self.build_H(improved=improved) #builds H and gets the effective coupling
        # Smallest algebraic eigenvalues
        evals, evecs = eigsh(H, k=n_eigs, which="SA") #gets smallest eigenvalues, gives ground state and low excitations
        evals.sort() #eigsh does not guarantee sorted output so we make sure they are
        return evals, lam_eff

##### Run it

In [9]:
if __name__ == "__main__":
    # Example parameters
    R = 10.0 / (2.0 * pi) # so L = 10
    mQ = 1.0 #need to look more into this but 1 should be fine for now?
    lam = 32 #coupling      
    kmax = 2 # keep small and increase carefully, this will probably be how we combine with digitisaiton

    params = Params(R=R, mQ=mQ, lam=lam, kmax=kmax)

    Emax_values = [6.0, 7.0, 8.0, 9.0, 10.0]  #energy cut offs to test

    print("Emax | dim | E0_raw | E0_imp | lam_eff") #table header
    for Emax in Emax_values: #does each energy cut off
        solver = HTETSolver(params, Emax, enforce_p0_sector=True) #contructs the solver
        dim = len(solver.basis) #dimension of basis

        ev_raw, lam_raw = solver.solve_lowest(n_eigs=3, improved=False) #without improvements
        ev_imp, lam_eff = solver.solve_lowest(n_eigs=3, improved=True) #with improvements

        #prints table of results 
        print(f"{Emax:4.1f} | {dim:4d} | {ev_raw[0]: .6f} {ev_raw[1]: .6f} |"f" {ev_imp[0]: .6f} {ev_imp[1]: .6f} | {lam_eff: .6f}")


Emax | dim | E0_raw | E0_imp | lam_eff
 6.0 |   23 |  0.140312  1.272396 |  0.140312  1.272396 |  32.000000
 7.0 |   33 |  0.140275  1.272243 |  0.140275  1.272243 |  32.000000
 8.0 |   49 |  0.140243  1.272135 |  0.140243  1.272135 |  32.000000
 9.0 |   68 |  0.140235  1.272058 |  0.140235  1.272058 |  32.000000
10.0 |   91 |  0.140235  1.272040 |  0.140235  1.272040 |  32.000000
